In [ ]:
from __future__ import annotations
import os, sys, csv, glob, pathlib, datetime as dt
from pathlib import Path
from typing import List, Set, Dict, Optional, Tuple
import pandas as pd
import duckdb

_ANALYTICS_SHARED = Path.cwd().resolve().parents[2] / "Analytics" / "shared"
if str(_ANALYTICS_SHARED) not in sys.path:
    sys.path.insert(0, str(_ANALYTICS_SHARED))

from locations import Location

# Tweak DuckDB defaults for low memory
DUCK_THREADS = min(8, os.cpu_count() or 1)
DUCK_MEM = "8GB"  # adjust if needed

def get_duck(threads: int = DUCK_THREADS, mem: str = DUCK_MEM) -> duckdb.DuckDBPyConnection:
    con = duckdb.connect()
    con.execute(f"PRAGMA threads={threads};")
    con.execute(f"PRAGMA memory_limit='{mem}';")
    # Parquet settings that often help with big scans
    con.execute("PRAGMA enable_progress_bar=false;")
    return con


In [ ]:
import pandas as pd
import os

def load_token_whitelist_from_status_csv(csv_path: str) -> Set[str]:
    """
    Build a whitelist of contract addresses from a token status CSV.
    Keeps rows where:
      - is_erc_20 == TRUE
      - local_status == PASS
      - price_data_status == success
    Returns lowercased contract addresses.
    """
    if not os.path.exists(csv_path):
        print(f"Token status CSV not found: {csv_path}")
        return set()

    try:
        df = pd.read_csv(csv_path, dtype=str)

        def norm(s):
            return str(s).strip().lower()

        mask = (
            df.get("is_erc_20", "").map(norm) == "true"
        ) & (
            df.get("local_status", "").map(norm) == "pass"
        ) & (
            df.get("price_data_status", "").map(norm) == "success"
        )

        if "contract_address" not in df.columns:
            print("'contract_address' column missing in token status CSV")
            return set()

        addrs = (
            df.loc[mask, "contract_address"]
              .dropna()
              .astype(str)
              .str.strip()
              .str.lower()
              .tolist()
        )

        whitelist = {a for a in addrs if a.startswith("0x") and len(a) == 42}
        print(f"Token whitelist built: {len(whitelist)} ERC-20 tokens")
        return whitelist

    except Exception as e:
        print(f"Error reading token status CSV: {e}")
        return set()


In [ ]:
from typing import List

def resolve_blocks_for_dates(
    dates: List[str] | List[pd.Timestamp],
    block_to_date_csv: str,
) -> List[int]:
    if not dates:
        return []
    m = pd.read_csv(block_to_date_csv)
    block_col = next((c for c in ["block_number", "block"] if c in m.columns), None)
    if block_col is None:
        raise ValueError(
            f"Could not find a block column in {block_to_date_csv}. "
            f"Expected one of ['block_number','block']; got {list(m.columns)}"
        )
    date_col = next((c for c in ["date", "bound_date", "block_time_utc"] if c in m.columns), None)
    if date_col is None:
        raise ValueError(
            f"Expected a date-like column ('date' or 'bound_date' or 'block_time_utc') "
            f"in {block_to_date_csv}. Got: {list(m.columns)}"
        )
    m[date_col] = pd.to_datetime(m[date_col], errors="coerce").dt.normalize()
    m = m.dropna(subset=[date_col, block_col])

    out_blocks: List[int] = []
    for d in dates:
        d_norm = pd.to_datetime(d, errors="coerce")
        if pd.isna(d_norm):
            print(f"[blocks] WARN could not parse date '{d}' (skip)")
            continue
        d_norm = d_norm.normalize()
        row = m.loc[m[date_col] == d_norm]
        if row.empty:
            print(f"[blocks] WARN no block for date {d_norm.date()} (skip)")
            continue
        block_num = int(row.iloc[0][block_col])
        out_blocks.append(block_num)
    if not out_blocks:
        print("[blocks] WARN no valid dates/blocks resolved.")
    return out_blocks


In [ ]:
def month_starts(start: str, end: str) -> List[pd.Timestamp]:
    """
    Inclusive list of month-start dates from start to end (YYYY-MM-DD).
    """
    s = pd.Timestamp(start).normalize()
    e = pd.Timestamp(end).normalize()
    if s.day != 1:
        s = pd.Timestamp(year=s.year, month=s.month, day=1)
    # ensure month-start for end
    e = pd.Timestamp(year=e.year, month=e.month, day=1)
    rng = pd.date_range(start=s, end=e, freq="MS")  # Month Start
    return list(rng)

def find_token_parquet_glob(data_root: str | pathlib.Path, token_addr: str) -> Optional[str]:
    """
    Returns a glob that matches parquet(s) for the token.
    Supports:
      - {root}/token_address=<addr>/*.parquet
      - {root}/token_address=<addr>.parquet
      - direct shards under a folder with the exact address name
    """
    root = pathlib.Path(data_root)
    addr = token_addr.lower()
    candidates = [
        str(root / f"token_address={addr}" / "*.parquet"),
        str(root / f"token_address={addr}.parquet"),
        str(root / addr / "*.parquet"),
        str(root / addr / "**" / "*.parquet"),
    ]
    for g in candidates:
        matches = glob.glob(g, recursive=True)
        if matches:
            # return the glob pattern (DuckDB can expand wildcards)
            return g
    return None

def detect_transfer_schema(con, parquet_glob: str):
    cols = list(con.execute(f"SELECT * FROM read_parquet('{parquet_glob}') LIMIT 0").df().columns)
    lower = {c.lower(): c for c in cols}

    def pick(cands):
        for c in cands:
            if c in lower:
                return lower[c]
        return None

    # Try edge schema
    block_edge = pick(["block_number", "block", "blk_num"])
    from_col   = pick(["from_address", "from", "src"])
    to_col     = pick(["to_address", "to", "dst"])
    if block_edge and from_col and to_col:
        return {"mode": "edge", "block": block_edge, "from": from_col, "to": to_col}

    # Try delta schema (your files)
    block_delta   = pick(["block_number", "block", "blk_num"])
    address_col   = pick(["address", "addr"])
    value_col     = pick(["value", "amount", "delta"])
    if block_delta and address_col and value_col:
        return {"mode": "delta", "block": block_delta, "address": address_col, "value": value_col}

    raise ValueError(f"Unsupported schema. Found columns: {cols}")



In [ ]:
def count_wallets_acquired_up_to_blocks_for_token(
    con,
    parquet_glob: str,
    schema: dict,
    target_blocks: list[int],
):
    """
    Counts unique wallets that have acquired at least one asset for this token
    on or before each target block.

    - For delta schema (address, block, value):
        * treat rows where value > 0 (as integer) as 'acquisition'
        * we implement 'value > 0' with string logic to avoid overflow
    - For edge schema (from/to):
        * treat 'to' addresses as acquisitions
    """
    vals = ", ".join(f"({int(b)})" for b in target_blocks)
    targets_cte = f"SELECT * FROM (VALUES {vals}) AS t(block)"

    if schema["mode"] == "edge":
        block_col = schema["block"]
        to_col    = schema["to"]

        query = f"""
        WITH gains AS (
            SELECT {block_col} AS block_number, {to_col} AS addr
            FROM read_parquet('{parquet_glob}')
        ),
        first_seen AS (
            SELECT lower(addr) AS addr, MIN(block_number) AS first_seen_block
            FROM gains
            WHERE addr IS NOT NULL AND addr <> ''
            GROUP BY addr
        ),
        targets AS (
            {targets_cte}
        )
        SELECT t.block AS block, CAST(COUNT(*) AS BIGINT) AS wallets
        FROM first_seen fs
        JOIN targets t
          ON fs.first_seen_block <= t.block
        GROUP BY t.block
        ORDER BY t.block
        """
        return con.execute(query).df()

    elif schema["mode"] == "delta":
        block_col   = schema["block"]
        address_col = schema["address"]
        value_col   = schema["value"]

        # We detect value > 0 via string rules:
        #   val_str = TRIM(CAST(value AS VARCHAR))
        #   positive if:
        #     - not empty
        #     - not all zeros
        #     - first char is not '-'
        #
        # This avoids any HUGEINT/INT overflow issues.
        query = f"""
        WITH deltas AS (
            SELECT
                {block_col}   AS block_number,
                {address_col} AS addr
            FROM read_parquet('{parquet_glob}')
            WHERE {address_col} IS NOT NULL
              AND {address_col} <> ''
              AND {value_col} IS NOT NULL
              AND regexp_replace(TRIM(CAST({value_col} AS VARCHAR)), '^0+', '') <> ''
              AND substr(TRIM(CAST({value_col} AS VARCHAR)), 1, 1) <> '-'
        ),
        first_seen AS (
            SELECT lower(addr) AS addr, MIN(block_number) AS first_seen_block
            FROM deltas
            WHERE addr IS NOT NULL AND addr <> ''
            GROUP BY addr
        ),
        targets AS (
            {targets_cte}
        )
        SELECT t.block AS block, CAST(COUNT(*) AS BIGINT) AS wallets
        FROM first_seen fs
        JOIN targets t
          ON fs.first_seen_block <= t.block
        GROUP BY t.block
        ORDER BY t.block
        """
        return con.execute(query).df()

    else:
        raise ValueError(f"Unknown schema mode: {schema['mode']}")


In [ ]:
def count_wallets_up_to_blocks_for_token(
    con,
    parquet_glob: str,
    schema: dict,
    target_blocks: list[int],
):
    vals = ", ".join(f"({int(b)})" for b in target_blocks)
    targets_cte = f"SELECT * FROM (VALUES {vals}) AS t(block)"

    if schema["mode"] == "edge":
        block_col = schema["block"]; from_col = schema["from"]; to_col = schema["to"]
        query = f"""
        WITH transfers AS (
            SELECT {block_col} AS block_number, {from_col} AS addr
            FROM read_parquet('{parquet_glob}')
            UNION ALL
            SELECT {block_col} AS block_number, {to_col} AS addr
            FROM read_parquet('{parquet_glob}')
        ),
        first_seen AS (
            SELECT lower(addr) AS addr, MIN(block_number) AS first_seen_block
            FROM transfers
            WHERE addr IS NOT NULL AND addr <> ''
            GROUP BY addr
        ),
        targets AS (
            {targets_cte}
        )
        SELECT t.block AS block, CAST(COUNT(*) AS BIGINT) AS wallets
        FROM first_seen fs
        JOIN targets t
          ON fs.first_seen_block <= t.block
        GROUP BY t.block
        ORDER BY t.block
        """
        return con.execute(query).df()

    elif schema["mode"] == "delta":
        block_col = schema["block"]; address_col = schema["address"]
        query = f"""
        WITH seen AS (
            SELECT {block_col} AS block_number, {address_col} AS addr
            FROM read_parquet('{parquet_glob}')
        ),
        first_seen AS (
            SELECT lower(addr) AS addr, MIN(block_number) AS first_seen_block
            FROM seen
            WHERE addr IS NOT NULL AND addr <> ''
            GROUP BY addr
        ),
        targets AS (
            {targets_cte}
        )
        SELECT t.block AS block, CAST(COUNT(*) AS BIGINT) AS wallets
        FROM first_seen fs
        JOIN targets t
          ON fs.first_seen_block <= t.block
        GROUP BY t.block
        ORDER BY t.block
        """
        return con.execute(query).df()
    else:
        raise ValueError(f"Unknown schema mode: {schema['mode']}")


In [ ]:
PORTFOLIO_RECONSTRUCTION_DATA = Location.PORTFOLIO_RECONSTRUCTION_DATA

TOKEN_STATUS_CSV = Location.TOKEN_STATUS_CSV
BLOCK_TO_DATE_CSV = Location.BLOCK_TO_DATE_CSV

OUTPUT_CSV = "output_wallet_counts.csv"  

START_DATE = "2020-03-01"
END_DATE   = "2025-03-01"

# DuckDB tuning (optional overrides)
DUCK_THREADS = min(8, os.cpu_count() or 1)
DUCK_MEM = "8GB"

# Sanity prints
print("Data root:", PORTFOLIO_RECONSTRUCTION_DATA)
print("Status CSV:", TOKEN_STATUS_CSV)
print("Block map CSV:", BLOCK_TO_DATE_CSV)
print("Output CSV:", OUTPUT_CSV)


In [ ]:
dates = month_starts(START_DATE, END_DATE)
date_strs = [d.date().isoformat() for d in dates]
blocks = resolve_blocks_for_dates(dates, BLOCK_TO_DATE_CSV)

# Pair back to dates, but filter out any dates that failed to resolve
date_block_pairs = [(d, b) for d, b in zip(dates, blocks) if pd.notna(b)]
if not date_block_pairs:
    raise RuntimeError("No dates could be resolved to blocks. Check BLOCK_TO_DATE_CSV.")

print(f"Resolved {len(date_block_pairs)} monthly blocks from {date_block_pairs[0][0].date()} "
      f"to {date_block_pairs[-1][0].date()}.")


In [ ]:
whitelist = load_token_whitelist_from_status_csv(TOKEN_STATUS_CSV)
if not whitelist:
    print("WARN: Whitelist is empty. Nothing to do.")
len(whitelist)


In [ ]:
con = get_duck(DUCK_THREADS, DUCK_MEM)

# Output CSV header
os.makedirs(pathlib.Path(OUTPUT_CSV).parent, exist_ok=True)
with open(OUTPUT_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    # token_address, block, date, wallets_with_>=1_txn_up_to_block
    w.writerow(["token_address", "block", "date", "wallets"])

target_blocks = [int(b) for _, b in date_block_pairs]
date_map = {int(b): d.date().isoformat() for d, b in date_block_pairs}

processed = 0
skipped = 0
errors = 0

for token in sorted(whitelist):
    g = find_token_parquet_glob(PORTFOLIO_RECONSTRUCTION_DATA, token)
    if not g:
        skipped += 1
        if skipped % 50 == 0:
            print(f"[skip] {skipped} tokens had no parquet found so far...")
        continue

    try:
        # detect schema
        schema = detect_transfer_schema(con, g)
        df_counts = count_wallets_up_to_blocks_for_token(
            con=con,
            parquet_glob=g,
            schema=schema,
            target_blocks=target_blocks,
        )
        
        # append to CSV
        with open(OUTPUT_CSV, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for _, row in df_counts.iterrows():
                b = int(row["block"])
                w.writerow([token, b, date_map.get(b, ""), int(row["wallets"])])

        processed += 1
        if processed % 25 == 0:
            print(f"[ok] processed {processed} tokens...")

    except Exception as e:
        errors += 1
        print(f"[err] token {token}: {e}")

print(f"Done. processed={processed}, skipped(no parquet)={skipped}, errors={errors}")
print(f"Wrote: {OUTPUT_CSV}")


In [ ]:
OUTPUT_CSV_ACQUIRED = "output_wallet_counts_acquired.csv"  # new file
print("Acquired-output CSV:", OUTPUT_CSV_ACQUIRED)


In [ ]:
con = get_duck(DUCK_THREADS, DUCK_MEM)

# Initialize CSV for 'acquired at least one asset' metric
os.makedirs(pathlib.Path(OUTPUT_CSV_ACQUIRED).parent, exist_ok=True)
with open(OUTPUT_CSV_ACQUIRED, "w", newline="", encoding="utf-8") as f:
    w = csv.writer(f)
    # same schema as before for convenience
    w.writerow(["token_address", "block", "date", "wallets"])

target_blocks = [int(b) for _, b in date_block_pairs]
date_map = {int(b): d.date().isoformat() for d, b in date_block_pairs}

processed = 0
skipped = 0
errors = 0

for token in sorted(whitelist):
    g = find_token_parquet_glob(PORTFOLIO_RECONSTRUCTION_DATA, token)
    if not g:
        skipped += 1
        if skipped % 50 == 0:
            print(f"[acq/skip] {skipped} tokens had no parquet found so far...")
        continue

    try:
        schema = detect_transfer_schema(con, g)

        df_counts = count_wallets_acquired_up_to_blocks_for_token(
            con=con,
            parquet_glob=g,
            schema=schema,
            target_blocks=target_blocks,
        )

        # append to CSV
        with open(OUTPUT_CSV_ACQUIRED, "a", newline="", encoding="utf-8") as f:
            w = csv.writer(f)
            for _, row in df_counts.iterrows():
                b = int(row["block"])
                w.writerow([token, b, date_map.get(b, ""), int(row["wallets"])])

        processed += 1
        if processed % 25 == 0:
            print(f"[acq/ok] processed {processed} tokens...")

    except Exception as e:
        errors += 1
        print(f"[acq/err] token {token}: {e}")

print(f"[acq] Done. processed={processed}, skipped(no parquet)={skipped}, errors={errors}")
print(f"[acq] Wrote: {OUTPUT_CSV_ACQUIRED}")
